# Dataset Curation: Verified PET Positives & Biological Negative Controls

## Why This Step Is Needed
In the raw PAZy snapshot, all 485 entries are plastic-active enzymes: `reported_on_pet = 0` means PET activity is **not reported**, not experimentally disproved.

To build a valid binary classification dataset before sequence clustering, we:
1. Retain only **verified PET-degrading enzymes** from PAZy (`reported_on_pet == 1`).
2. Exclude unverified non-PET PAZy entries from being falsely labeled as negatives.
3. Retrieve **reviewed non-plastic carboxylic ester hydrolases (`EC 3.1.1.*`)** from UniProt Swiss-Prot within the same sequence length range (150–800 amino acids) as biological hard negatives.
4. Remove any sequence or accession overlap with PAZy, filter non-canonical amino acids, and save `data/processed/curated/curated_pet_dataset.csv`.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

## 1. Load PAZy Positives & Fetch UniProt Swiss-Prot Negative Controls

In [ ]:
from src.data import (
    load_pazy_dataset,
    fetch_uniprot_negative_hydrolases,
    build_curated_pet_dataset,
)

pazy_dataset = load_pazy_dataset()
negative_controls = fetch_uniprot_negative_hydrolases(
    max_records=400,
    min_length=150,
    max_length=800,
)

curated_dataset, curation_summary = build_curated_pet_dataset(
    pazy_dataset=pazy_dataset,
    negative_controls=negative_controls,
    max_negatives=317,
    random_seed=42,
)

display(curation_summary)

## 2. Inspect Class Balance & Sequence Length Distributions

In [ ]:
from src.visualizations import plot_curated_dataset_balance

plot_curated_dataset_balance(curated_dataset)

## 3. Save Curated Dataset

In [ ]:
curated_dir = project_root / "data" / "processed" / "curated"
curated_dir.mkdir(parents=True, exist_ok=True)

curated_csv_path = curated_dir / "curated_pet_dataset.csv"
curated_dataset.to_csv(curated_csv_path, index=False)
print(f"Saved curated dataset ({len(curated_dataset)} rows) to: {curated_csv_path}")
curated_dataset.drop(columns=["sequence"]).head(10)